In [1]:
import os
import sys
# Agregar el directorio raíz del proyecto al path
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)
    
    
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *   
from services.utilidades import *
from services.comprobar_netcdf import *

In [2]:
# Recargar módulos locales (ejecutar esta celda cuando cambies código en services/)
import importlib
import sys

# Lista de módulos locales a recargar
modulos_locales = [
    'services.carga_de_datos',
    'services.DataFrame_maker',
    'services.prueba',
    'services.utilidades',
    'services.get_tspan',
    'services.polar2uv',
    'services.uv2polar'
]

for modulo in modulos_locales:
    if modulo in sys.modules:
        importlib.reload(sys.modules[modulo])
        print(f"✓ Recargado: {modulo}")
    else:
        print(f"- No cargado: {modulo}")

# Re-importar después de recargar
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *

print("\n¡Módulos locales recargados!")

✓ Recargado: services.carga_de_datos
✓ Recargado: services.DataFrame_maker
- No cargado: services.prueba
✓ Recargado: services.utilidades
✓ Recargado: services.get_tspan
✓ Recargado: services.polar2uv
✓ Recargado: services.uv2polar

¡Módulos locales recargados!


In [3]:
# %matplotlib tk

In [4]:
dict_de_boyas = {}

### 2. OLEAJE

In [5]:
# 1.1. p1: Cargar crudo pkl 
nombre = "BMT3-10-T45_C"
archivos_csv = "msj3_crudo_unido_realt_2026041603_2026060923.pkl"
ruta_a_carpeta = "\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_4.2\\2026\\06. Junio\\BMT3-10-T45\\DATOS"
df_pkl_crudo_realt = cargar_pickle_oleaje(ruta_a_carpeta, archivos_csv)
dict_de_boyas[nombre] = df_pkl_crudo_realt # Guardo el dataframe dentro del diccionario de boyas.
df_pkl_crudo_realt.head(2)

,tspan,Hs,Hmax,Tp,dir,vardir
0,2026-04-16 03:00:00,0.93,1.66,4.4,36.0,32.9
1,2026-04-16 04:00:00,1.14,2.20,4.9,36.1,31.9


In [6]:
# 1.5. p1: Cargar NETCDF de corrientes 
nombre = "BMT3-10-T45_NC"
nombre_de_archivo = "BMT3-10-T45-TRIAXYSG3-NS09164-Z0-REALT-20260416-20260609.nc"
ruta_a_carpeta = "\\\\192.168.15.249\\Med_2025-2026\\Reportes_Edit\\Reporte_4.2\\2026\\06. Junio\\BMT3-10-T45\\DATOS"
df_nc, nc, DirSpec = cargar_nc_oleaje(ruta_a_carpeta, nombre_de_archivo)
dict_de_boyas[nombre] = df_nc # Guardo el dataframe dentro del diccionario de boyas.
df_nc.tail(2)


,tspan,Hs,Hmax,Tp,dir,vardir
1315,2026-06-09 22:00:00,0.73,1.17,5.4,45.1,36.1
1316,2026-06-09 23:00:00,0.78,1.17,4.9,40.9,32.6


In [7]:
# Otras revisiones del NC:
# comprobar_nombre_nc
comprobar_nombre_nc(nombre_de_archivo, nc)
comprobar_porcentaje_de_variable(nc, ruta_a_carpeta, "BMT3-10-T45_datos_word_2026041603_2026060923.xlsx", variables="oleaje")

NAM correcto


,Variable,esperados,validados,porcentaje_validados
0,Hs,1317,1315.0,99.84814
1,Hm,1317,1315.0,99.84814
2,Tp,1317,1315.0,99.84814
3,dir,1317,1315.0,99.84814
4,vardir,1317,1315.0,99.84814
5,dirspec,1317,1177.0,89.36978


In [8]:
# Crear el diccionario de boyas con los dataframes cargados
df = crear_dataframe_de_boyas(dict_de_boyas)

Index(['tspan', 'Hs_BMT3-10-T45_C', 'Hmax_BMT3-10-T45_C', 'Tp_BMT3-10-T45_C',
       'dir_BMT3-10-T45_C', 'vardir_BMT3-10-T45_C', 'Hs_BMT3-10-T45_NC',
       'Hmax_BMT3-10-T45_NC', 'Tp_BMT3-10-T45_NC', 'dir_BMT3-10-T45_NC',
       'vardir_BMT3-10-T45_NC'],
      dtype='str')


In [ ]:
app = InteractivePlotterTk(df)
# Iniciar el bucle de la aplicación
app.root.mainloop()

# NOTAS PENDIENTES:
# - Arreglar que al eliminar puntos para que no tenga límite el historial de deshacer
# - Arreglar para que solo se grafiquen las columnas elegidas por el usuario
# - Agregar una función que grafique las variables de Rap y Dir --> sería algo como que df.iloc[idx,col][prof]
# - Arreglar el caso de cuando el usuario elimina puntos de Rap o Dir colocar U y V como nans --> df.iloc[idx,col][prof] = np.nan
# - Agregar la funcionalidad de guardar el DataFrame corregido en un archivo .csv o .xlsx, pkl
# - Agregar la funcionalidad de devolver puntos eliminados (no deshacer la última acción, sino devolver puntos específicos)

In [ ]:
# from services.guardar_puntos_eliminados import guardar_puntos_eliminados
# nombre_de_archivo = "borrar_puntos"
# guardar_puntos_eliminados(app, ruta_a_carpeta, nombre_de_archivo)

In [ ]:
from pickle import load
ruta = os.path.join(ruta_a_carpeta, "borrar_puntos.pkl")
with open(ruta, 'rb') as f:
    puntos_eliminados = load(f)
    
puntos_eliminados